# 06. Experimentos sobre el empate

El filtro de cuotas del backtest principal excluye el empate (X). Aquí investigo si hay rangos de cuota donde apostar al empate sí tenga edge real.

Estructura:

- Experimento 1: 7 configuraciones base con/sin X en distintos rangos.
- Experimento 2: granularidad fina, 13 sub-rangos de X entre 3.20 y 5.00.
- Experimento 3: validación en dos fases (in-sample 2015-2019 / OOS 2020-2024), EV mínimo variable y kill switch.

Conclusión por adelantado: el hit rate real del empate (25-26%) es prácticamente igual a la probabilidad implícita (~26%). No hay edge probabilístico claro. Algunos sub-rangos puntuales dan ROI positivo, pero proceden de varianza, no de aciertos extra. Solo el sub-rango X[3.70-4.50) sobrevive in-sample y OOS de forma consistente, aunque con muestra pequeña.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import pandas as pd
import numpy as np
from IPython.display import display
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit

# ─── Config ───────────────────────────────────────────────────────────────────
DATA_PATH    = os.path.join(os.getcwd(), '..', 'data', 'df_final_clean.csv')
TRAIN_WINDOW = 5
FIRST_TEST   = 2015
MIN_EV       = 0.05
FLAT_STAKE   = 10.0
INIT_BK      = 1000.0

MODEL_FEATURES = [
    "Home_Elo_Calc", "Away_Elo_Calc", "Elo_Diff",
    "Home_Market_Value", "Away_Market_Value", "Log_Value_Diff",
    "Diff_FIFA_Ova", "Diff_FIFA_Mid", "Diff_FIFA_Def", "Diff_FIFA_Att",
    "Home_Streak_L5", "Away_Streak_L5",
    "Home_H2H_L3",   "Away_H2H_L3",
]

import json
with open('../results/best_params.json') as _f:
    _bp = json.load(_f)
XGB_PARAMS = {**_bp, 'objective':'multi:softprob', 'num_class':3, 'eval_metric':'mlogloss',
              'random_state':42, 'verbosity':0, 'n_jobs':-1}

# ─── Helpers ──────────────────────────────────────────────────────────────────

def build_elo(df, k=30, ha=100, start=1500):
    """Calcula ratings Elo acumulados partido a partido."""
    ratings = {}
    h_elos, a_elos = [], []
    for _, row in df.iterrows():
        h, a, ftr = row["HomeTeam"], row["AwayTeam"], row["FTR"]
        rh = ratings.get(h, start)
        ra = ratings.get(a, start)
        h_elos.append(rh)
        a_elos.append(ra)
        e_h = 1.0 / (1.0 + 10 ** ((ra - (rh + ha)) / 400.0))
        s_h, s_a = (1, 0) if ftr == "H" else ((0.5, 0.5) if ftr == "D" else (0, 1))
        ratings[h] = rh + k * (s_h - e_h)
        ratings[a] = ra + k * (s_a - (1 - e_h))
    return h_elos, a_elos


def passes_filter(bt, odds, filt):
    """Comprueba si una apuesta pasa el filtro de cuotas."""
    if filt is None:
        return True
    return any(bt == b and lo <= odds < hi for b, lo, hi in filt)


def passes_filter_ev(bt, odds, filt, min_ev_x=None, ev=None):
    """Filtra por rango de cuota. min_ev_x permite EV minimo diferente para X."""
    if filt is None:
        return True
    if bt == "X" and min_ev_x is not None:
        if ev is None or ev <= min_ev_x:
            return False
    return any(bt == b and lo <= odds < hi for b, lo, hi in filt)


def run_config(df_clean, all_seasons, test_seasons, odds_filter):
    """Walk-forward sliding-5 con filtro de cuotas simple."""
    all_bets, season_rows = [], []

    for test_s in test_seasons:
        prior = [s for s in all_seasons if s < test_s]
        tr_s  = prior[-TRAIN_WINDOW:]
        if not tr_s:
            continue

        tr_mask = df_clean["Season"].isin(tr_s)
        te_mask = df_clean["Season"] == test_s
        X_tr = df_clean.loc[tr_mask, MODEL_FEATURES].astype(float)
        y_tr = df_clean.loc[tr_mask, "Target"].astype(int)
        df_te = df_clean.loc[te_mask].reset_index(drop=True)
        if len(X_tr) < 50 or len(df_te) == 0:
            continue

        model = CalibratedClassifierCV(XGBClassifier(**XGB_PARAMS),
                                       method="isotonic", cv=TimeSeriesSplit(n_splits=3))
        model.fit(X_tr.values, y_tr.values)
        proba   = model.predict_proba(df_te[MODEL_FEATURES].astype(float).values)
        classes = list(model.classes_)
        p_H = proba[:, classes.index(2)]
        p_D = proba[:, classes.index(1)]
        p_A = proba[:, classes.index(0)]

        season_bets = []
        for i, row in df_te.iterrows():
            oh = float(row["B365H"])
            od = float(row["B365D"])
            oa = float(row["B365A"])
            if oh < 1.05 or od < 1.05 or oa < 1.05:
                continue
            ftr = str(row["FTR"])
            for bt, p, odds, won in [("1", p_H[i], oh, ftr=="H"),
                                     ("X", p_D[i], od, ftr=="D"),
                                     ("2", p_A[i], oa, ftr=="A")]:
                ev = p * odds - 1
                if ev <= MIN_EV:
                    continue
                if not passes_filter(bt, odds, odds_filter):
                    continue
                profit = FLAT_STAKE * (odds - 1) if won else -FLAT_STAKE
                season_bets.append({
                    "Season": test_s, "BetType": bt, "Odds": odds,
                    "P_Model": p, "Won": int(won),
                    "Flat_P": profit, "Flat_S": FLAT_STAKE,
                })

        if season_bets:
            b = pd.DataFrame(season_bets)
            fr = b["Flat_P"].sum() / b["Flat_S"].sum()
            season_rows.append({
                "Season":   test_s,
                "N_Bets":   len(b),
                "Hit_Rate": round(b["Won"].mean(), 3),
                "Flat_ROI": round(fr, 4),
            })
            all_bets.extend(season_bets)

    df_bets    = pd.DataFrame(all_bets)
    df_seasons = pd.DataFrame(season_rows)
    return df_bets, df_seasons


def run_wf(df_clean, all_seasons, test_seasons, odds_filter,
           min_ev=0.05, min_ev_x=None, kill_switch=False,
           kill_min=15, kill_thr=0.08):
    """Walk-forward con soporte para EV minimo variable en X y kill switch.
    Kill switch: para la temporada si hit_real < P_Model_medio - kill_thr.
    Umbral por defecto reducido a 0.08 (antes 0.12) para ser efectivo con
    la sobreconfianza sistematica observada de 8-12 pp.
    """
    all_bets, season_rows = [], []
    for test_s in test_seasons:
        prior = [s for s in all_seasons if s < test_s]
        tr_s  = prior[-TRAIN_WINDOW:]
        if not tr_s:
            continue
        tr = df_clean["Season"].isin(tr_s)
        te = df_clean["Season"] == test_s
        X_tr = df_clean.loc[tr, MODEL_FEATURES].astype(float)
        y_tr = df_clean.loc[tr, "Target"].astype(int)
        df_te = df_clean.loc[te].reset_index(drop=True)
        if len(X_tr) < 50 or len(df_te) == 0:
            continue

        model = CalibratedClassifierCV(XGBClassifier(**XGB_PARAMS),
                                       method="isotonic", cv=TimeSeriesSplit(n_splits=3))
        model.fit(X_tr.values, y_tr.values)
        proba   = model.predict_proba(df_te[MODEL_FEATURES].astype(float).values)
        classes = list(model.classes_)
        p_H = proba[:, classes.index(2)]
        p_D = proba[:, classes.index(1)]
        p_A = proba[:, classes.index(0)]

        season_bets = []
        killed = False
        for i, row in df_te.iterrows():
            if killed:
                break
            oh = float(row["B365H"])
            od = float(row["B365D"])
            oa = float(row["B365A"])
            if oh < 1.05 or od < 1.05 or oa < 1.05:
                continue
            ftr = str(row["FTR"])
            for bt, p, odds, won in [("1", p_H[i], oh, ftr=="H"),
                                     ("X", p_D[i], od, ftr=="D"),
                                     ("2", p_A[i], oa, ftr=="A")]:
                ev = p * odds - 1
                if ev <= min_ev:
                    continue
                if not passes_filter_ev(bt, odds, odds_filter, min_ev_x, ev):
                    continue
                profit = FLAT_STAKE * (odds - 1) if won else -FLAT_STAKE
                season_bets.append({
                    "Season": test_s, "BetType": bt, "Odds": odds,
                    "P_Model": p, "EV": ev, "Won": int(won),
                    "Flat_P": profit, "Flat_S": FLAT_STAKE,
                })
            # Kill switch: obs vs P_Model predicho (correcto)
            if kill_switch and len(season_bets) >= kill_min:
                obs = np.mean([b["Won"]     for b in season_bets])
                exp = np.mean([b["P_Model"] for b in season_bets])
                if obs < exp - kill_thr:
                    killed = True

        if season_bets:
            b  = pd.DataFrame(season_bets)
            fr = b["Flat_P"].sum() / b["Flat_S"].sum()
            season_rows.append({
                "Season":   test_s,
                "N_Bets":   len(b),
                "Hit_Rate": round(b["Won"].mean(), 3),
                "Flat_ROI": round(fr, 4),
            })
            all_bets.extend(season_bets)

    return pd.DataFrame(all_bets), pd.DataFrame(season_rows)


def summary(bets, seasons, label):
    """Imprime una linea resumen de una configuracion."""
    if len(bets) == 0:
        print(f"  {label}: sin apuestas")
        return
    roi = bets["Flat_P"].sum() / bets["Flat_S"].sum()
    pos = (seasons["Flat_ROI"] > 0).sum()
    hr  = bets["Won"].mean()
    bk  = 1000 + bets["Flat_P"].sum()
    print(f"  {label:<40} bets={len(bets):4}  ROI={roi*100:+6.2f}%  "
          f"pos={pos}/{len(seasons)}  hit={hr*100:.1f}%  BK={bk:.0f}EUR")


# ─── Carga de datos ────────────────────────────────────────────────────────────
print("Cargando datos...")
df = pd.read_csv(DATA_PATH)
df["Date"]   = pd.to_datetime(df["Date"])
df = df[df["FTR"].isin(["H", "D", "A"])].dropna(subset=["Season"]).copy()
df["Season"] = df["Season"].astype(int)
df = df.sort_values("Date").reset_index(drop=True)

df["Home_Elo_Calc"], df["Away_Elo_Calc"] = build_elo(df)
df["Elo_Diff"] = df["Home_Elo_Calc"] - df["Away_Elo_Calc"]

for col in MODEL_FEATURES:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

required = MODEL_FEATURES + ["Target", "B365H", "B365D", "B365A", "FTR", "Season"]
df_clean = df.dropna(subset=required).copy()

all_seasons = sorted(df_clean["Season"].unique())
test_all    = [s for s in all_seasons if s >= FIRST_TEST]

# Filtro base SIN visitante: ROI historico -17.55%, sin edge real
BASE  = [("1", 1.40, 1.70), ("1", 2.00, 2.50)]
X_SEG = ("X", 3.50, 4.50)

print(f"Partidos: {len(df_clean)} | Test seasons: {test_all}")
print(f"BASE filter (sin visitante): {BASE}")

Cargando datos...
Partidos: 5700 | Test seasons: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
BASE filter (sin visitante): [('1', 1.4, 1.7), ('1', 2.0, 2.5)]


## Experimento 1: exploración inicial con/sin empate

7 configuraciones: el filtro base sin empate, distintos rangos de empate, y solo X. Quiero ver si algún rango concreto del empate da ROI positivo.

In [2]:
CONFIGS_E1 = {
    "A  Sin X (actual)":          [("1", 1.40, 1.70), ("1", 2.00, 2.50)],
    "B  + X todos rangos":        [("1", 1.40, 1.70), ("1", 2.00, 2.50),
                                   ("X", 2.50, 4.50)],
    "C  + X[2.50-3.00)":          [("1", 1.40, 1.70), ("1", 2.00, 2.50),
                                   ("X", 2.50, 3.00)],
    "D  + X[3.00-3.50)":          [("1", 1.40, 1.70), ("1", 2.00, 2.50),
                                   ("X", 3.00, 3.50)],
    "E  + X[3.50-4.50)":          [("1", 1.40, 1.70), ("1", 2.00, 2.50),
                                   ("X", 3.50, 4.50)],
    "F  Solo X todos rangos":     [("X", 2.50, 4.50)],
    "G  Sin filtro (solo EV>5%)": None,
}

results_e1 = []
for name, filt in CONFIGS_E1.items():
    print(f"  Ejecutando: {name}...")
    bets, seasons = run_config(df_clean, all_seasons, test_all, filt)
    if len(bets) == 0:
        results_e1.append({"Config": name, "Bets": 0, "ROI": "---",
                            "Pos": "---", "HitRate": "---"})
        continue
    roi = bets["Flat_P"].sum() / bets["Flat_S"].sum()
    pos = (seasons["Flat_ROI"] > 0).sum()
    hr  = bets["Won"].mean()
    results_e1.append({
        "Config":  name,
        "Bets":    len(bets),
        "ROI":     f"{roi*100:+.2f}%",
        "Pos":     f"{pos}/{len(seasons)}",
        "HitRate": f"{hr*100:.1f}%",
    })

print()
display(pd.DataFrame(results_e1))

  Ejecutando: A  Sin X (actual)...
  Ejecutando: B  + X todos rangos...
  Ejecutando: C  + X[2.50-3.00)...
  Ejecutando: D  + X[3.00-3.50)...
  Ejecutando: E  + X[3.50-4.50)...
  Ejecutando: F  Solo X todos rangos...
  Ejecutando: G  Sin filtro (solo EV>5%)...



,Config,Bets,ROI,Pos,HitRate
0,A Sin X (actual),449,-3.71%,4/10,52.8%
1,B + X todos rangos,1103,-2.91%,5/10,37.4%
2,C + X[2.50-3.00),452,-4.35%,4/10,52.4%
3,D + X[3.00-3.50),663,-5.32%,3/10,44.8%
4,E + X[3.50-4.50),886,-1.18%,5/10,39.7%
5,F Solo X todos rangos,654,-2.37%,5/10,26.8%
6,G Sin filtro (solo EV>5%),3394,-7.62%,1/10,30.2%


**Hallazgo:** el baseline sin X (449 bets, FIRST_TEST=2015) queda en -3.71% (4/10 temporadas). Añadir empate en rangos amplios deja números similares (B + X todos: -2.91%, 5/10) o claramente peores (D + X[3.00-3.50): -5.32%, 3/10). La menos mala con empate es E (BASE + X[3.50-4.50)) con -1.18% (886 bets, 5/10), por debajo del baseline pero la opción más resistente del bloque. "Solo X" (config F) pierde -2.37% (5/10) y abrir el filtro a todo con solo EV>5% (G) destroza el ROI (-7.62%, 3394 bets). Ningún rango aporta edge claro. Voy al Experimento 2 a granularidad fina.

## Experimento 2: granularidad fina del rango X[3.50-4.50)

13 sub-configuraciones alrededor del rango menos malo del Exp 1, moviendo los límites con paso 0.10.

In [3]:
BASE_FILTER = [("1", 1.40, 1.70), ("1", 2.00, 2.50)]

CONFIGS_E2 = {
    "REF  Sin X":         BASE_FILTER,
    "X[3.20-3.60)":       BASE_FILTER + [("X", 3.20, 3.60)],
    "X[3.30-3.70)":       BASE_FILTER + [("X", 3.30, 3.70)],
    "X[3.40-3.80)":       BASE_FILTER + [("X", 3.40, 3.80)],
    "X[3.50-4.00)":       BASE_FILTER + [("X", 3.50, 4.00)],
    "X[3.50-4.50)":       BASE_FILTER + [("X", 3.50, 4.50)],
    "X[3.50-5.00)":       BASE_FILTER + [("X", 3.50, 5.00)],
    "X[3.60-4.50)":       BASE_FILTER + [("X", 3.60, 4.50)],
    "X[3.70-4.50)":       BASE_FILTER + [("X", 3.70, 4.50)],
    "X[4.00-6.00)":       BASE_FILTER + [("X", 4.00, 6.00)],
    "Solo X[3.50-4.50)": [("X", 3.50, 4.50)],
    "Solo X[3.50-5.00)": [("X", 3.50, 5.00)],
    "Solo X[3.40-4.60)": [("X", 3.40, 4.60)],
}

results_e2 = []
for name, filt in CONFIGS_E2.items():
    sys.stdout.write(f"  {name}...\n")
    sys.stdout.flush()
    bets, seasons = run_config(df_clean, all_seasons, test_all, filt)
    if len(bets) == 0:
        results_e2.append({"Config": name, "Bets": 0, "ROI": "---",
                            "Pos": "---", "HitRate": "---"})
        continue
    roi = bets["Flat_P"].sum() / bets["Flat_S"].sum()
    pos = (seasons["Flat_ROI"] > 0).sum()
    hr  = bets["Won"].mean()
    results_e2.append({
        "Config":  name,
        "Bets":    len(bets),
        "ROI":     f"{roi*100:+.2f}%",
        "Pos":     f"{pos}/{len(seasons)}",
        "HitRate": f"{hr*100:.1f}%",
    })

print()
display(pd.DataFrame(results_e2))

# Detalle por temporada y tipo de apuesta para BASE + X[3.50-4.50)
print("\nDetalle por temporada: BASE + X[3.50-4.50)")
bets_best, seas_best = run_config(df_clean, all_seasons, test_all,
                                   BASE_FILTER + [("X", 3.50, 4.50)])
if len(seas_best):
    display(seas_best[["Season", "N_Bets", "Hit_Rate", "Flat_ROI"]])
    print("\nBreakdown por tipo de apuesta:")
    for bt in ["1", "X", "2"]:
        sub = bets_best[bets_best["BetType"] == bt]
        if len(sub) == 0:
            continue
        r = sub["Flat_P"].sum() / sub["Flat_S"].sum()
        print(f"  {bt}  n={len(sub):3}  hit={sub['Won'].mean():.1%}  "
              f"ROI={r*100:+.2f}%  avg_odds={sub['Odds'].mean():.2f}")

  REF  Sin X...
  X[3.20-3.60)...
  X[3.30-3.70)...
  X[3.40-3.80)...
  X[3.50-4.00)...
  X[3.50-4.50)...
  X[3.50-5.00)...
  X[3.60-4.50)...
  X[3.70-4.50)...
  X[4.00-6.00)...
  Solo X[3.50-4.50)...
  Solo X[3.50-5.00)...
  Solo X[3.40-4.60)...



,Config,Bets,ROI,Pos,HitRate
0,REF Sin X,449,-3.71%,4/10,52.8%
1,X[3.20-3.60),703,-6.42%,4/10,43.2%
2,X[3.30-3.70),721,-9.91%,3/10,41.6%
3,X[3.40-3.80),729,-8.52%,4/10,41.6%
4,X[3.50-4.00),731,-6.25%,3/10,41.9%
5,X[3.50-4.50),886,-1.18%,5/10,39.7%
6,X[3.50-5.00),957,-3.63%,5/10,37.8%
7,X[3.60-4.50),807,-0.18%,5/10,41.1%
8,X[3.70-4.50),739,+3.16%,5/10,43.3%
9,X[4.00-6.00),757,-3.85%,3/10,40.2%



Detalle por temporada: BASE + X[3.50-4.50)


,Season,N_Bets,Hit_Rate,Flat_ROI
0,2015,63,0.429,-0.2057
1,2016,54,0.481,0.1124
2,2017,88,0.500,0.0715
3,2018,77,0.455,0.0477
4,2019,64,0.375,-0.2466
5,2020,103,0.515,0.2411
6,2021,109,0.312,-0.2483
7,2022,110,0.318,-0.0488
8,2023,122,0.328,-0.0291
9,2024,96,0.354,0.1398



Breakdown por tipo de apuesta:
  1  n=449  hit=52.8%  ROI=-3.71%  avg_odds=1.91
  X  n=437  hit=26.3%  ROI=+1.42%  avg_odds=3.83


**Observación:** el único sub-rango con ROI positivo a granularidad fina es X[3.70-4.50): +3.16% (739 bets, 5/10). El sub-rango aislado Solo X[3.50-4.50) queda en +1.42% (437 bets, 5/10) con hit rate real 26.3%, casi igual a la implícita media (~26%). El modelo no acierta el empate más que el mercado; cualquier yield positivo viene de la combinación de cuotas altas (avg 3.83) y un excedente muy pequeño de aciertos en un bin estrecho. El Experimento 3 valida esto en out-of-sample y con kill switch.

## Experimento 3: validación out-of-sample (2 fases) y robustez

Para evitar sesgo in-sample, valido en dos fases:

- Fase 1 (2015-2019): exploración libre, identifico rangos prometedores.
- Fase 2 (2020-2024): evalúo el filtro derivado en la Fase 1.

Además compruebo el efecto del EV mínimo aplicado a las apuestas X y el efecto del kill switch.

In [4]:
phase1_seasons = [s for s in test_all if s <= 2019]
phase2_seasons = [s for s in test_all if s >= 2020]

print("=" * 65)
print("TEST 1: VALIDACION 2 FASES (fase1=2015-2019, fase2=2020-2024)")
print("=" * 65)

# Fase 1: sin filtro de cuotas, solo EV>5%
print("\nFase 1 (2015-2019) -- exploracion libre sin filtro de cuotas:")
bets1, seas1 = run_wf(df_clean, all_seasons, phase1_seasons,
                       odds_filter=None, min_ev=0.05)

# Breakdown por tipo y rango de cuota
bins = [1.0, 1.30, 1.50, 1.70, 2.00, 2.50, 3.00, 3.50, 4.00, 4.50, 5.00, 20.0]
for bt in ["1", "X", "2"]:
    sub = bets1[bets1["BetType"] == bt] if len(bets1) else pd.DataFrame()
    if len(sub) == 0:
        continue
    segs = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        seg = sub[(sub["Odds"] >= lo) & (sub["Odds"] < hi)]
        if len(seg) >= 5:
            sr = seg["Flat_P"].sum() / seg["Flat_S"].sum()
            segs.append(f"{lo:.2f}-{hi:.2f}:{sr*100:+.0f}%({len(seg)})")
    print(f"  {bt}: " + "  ".join(segs))

TEST 1: VALIDACION 2 FASES (fase1=2015-2019, fase2=2020-2024)

Fase 1 (2015-2019) -- exploracion libre sin filtro de cuotas:
  1: 1.00-1.30:-1%(28)  1.30-1.50:+10%(57)  1.50-1.70:-17%(72)  1.70-2.00:-13%(90)  2.00-2.50:-2%(149)  2.50-3.00:-29%(95)  3.00-3.50:+7%(65)  3.50-4.00:-23%(43)  4.00-4.50:+19%(28)  4.50-5.00:+47%(22)  5.00-20.00:+24%(113)
  X: 3.00-3.50:-7%(56)  3.50-4.00:-49%(43)  4.00-4.50:+42%(44)  4.50-5.00:-24%(25)  5.00-20.00:-26%(135)
  2: 1.50-1.70:-9%(14)  1.70-2.00:-36%(14)  2.00-2.50:-13%(33)  2.50-3.00:-8%(45)  3.00-3.50:-21%(69)  3.50-4.00:+14%(68)  4.00-4.50:+14%(57)  4.50-5.00:-37%(52)  5.00-20.00:-17%(200)


In [5]:
# Fase 2: evaluacion out-of-sample con los filtros derivados en fase 1
print("\nFase 2 (2020-2024) -- evaluación out-of-sample:")
configs_f2 = {
    "BASE sin X":           BASE,
    "BASE + X[3.50-4.50)": BASE + [X_SEG],
    "Solo X[3.50-4.50)":   [X_SEG],
    "BASE + X[3.40-4.60)": BASE + [("X", 3.40, 4.60)],
    "BASE + X[3.70-4.50)": BASE + [("X", 3.70, 4.50)],
}
for name, filt in configs_f2.items():
    b2, s2 = run_wf(df_clean, all_seasons, phase2_seasons, filt)
    summary(b2, s2, name)


Fase 2 (2020-2024) -- evaluación out-of-sample:
  BASE sin X                               bets= 190  ROI= -3.42%  pos=2/5  hit=53.7%  BK=935EUR
  BASE + X[3.50-4.50)                      bets= 540  ROI= +0.42%  pos=2/5  hit=36.3%  BK=1023EUR
  Solo X[3.50-4.50)                        bets= 350  ROI= +2.50%  pos=4/5  hit=26.9%  BK=1088EUR
  BASE + X[3.40-4.60)                      bets= 627  ROI= -4.31%  pos=2/5  hit=33.8%  BK=730EUR
  BASE + X[3.70-4.50)                      bets= 419  ROI= +4.95%  pos=2/5  hit=39.9%  BK=1208EUR


In [6]:
print("=" * 65)
print("TEST 3: CON KILL SWITCH (igual que backtest_master.py)")
print("=" * 65)

configs_ks = {
    "BASE sin X + KS":          (BASE,           0.05, None),
    "BASE + X[3.50-4.50) + KS": (BASE + [X_SEG], 0.05, None),
    "Solo X[3.50-4.50) + KS":   ([X_SEG],         0.05, None),
}
for name, (filt, ev, ev_x) in configs_ks.items():
    b, s = run_wf(df_clean, all_seasons, test_all, filt,
                   min_ev=ev, min_ev_x=ev_x, kill_switch=True)
    summary(b, s, name)

# Detalle por temporada de la configuracion final
print()
print("=" * 65)
print("DETALLE POR TEMPORADA: BASE + X[3.50-4.50) con Kill Switch")
print("=" * 65)
bX, sX = run_wf(df_clean, all_seasons, test_all,
                 BASE + [X_SEG], kill_switch=True)
if len(sX):
    s_final = sX[["Season", "N_Bets", "Hit_Rate", "Flat_ROI"]].copy()
    display(s_final)
    roi_x = bX["Flat_P"].sum() / bX["Flat_S"].sum()
    print(f"\nROI total: {roi_x*100:+.2f}% | BK final: {1000+bX['Flat_P'].sum():.0f} EUR")
    print(f"Implied prob media (1/odds): {(1/bX['Odds']).mean():.1%}")
    print(f"Hit rate real:               {bX['Won'].mean():.1%}")
    print(f"P_Model media:               {bX['P_Model'].mean():.1%}")

TEST 3: CON KILL SWITCH (igual que backtest_master.py)
  BASE sin X + KS                          bets= 224  ROI= -0.64%  pos=3/10  hit=53.6%  BK=986EUR
  BASE + X[3.50-4.50) + KS                 bets= 279  ROI= -0.24%  pos=3/10  hit=40.9%  BK=993EUR
  Solo X[3.50-4.50) + KS                   bets= 169  ROI= +0.01%  pos=2/10  hit=25.4%  BK=1000EUR

DETALLE POR TEMPORADA: BASE + X[3.50-4.50) con Kill Switch


,Season,N_Bets,Hit_Rate,Flat_ROI
0,2015,15,0.467,-0.1613
1,2016,54,0.481,0.1124
2,2017,17,0.412,-0.0882
3,2018,15,0.400,0.2220
4,2019,15,0.200,-0.6533
5,2020,103,0.515,0.2411
6,2021,15,0.200,-0.3440
7,2022,15,0.200,-0.2300
8,2023,15,0.133,-0.6567
9,2024,15,0.267,-0.1820



ROI total: -0.24% | BK final: 993 EUR
Implied prob media (1/odds): 42.1%
Hit rate real:               40.9%
P_Model media:               49.1%


## Conclusiones

El nicho del empate aporta señal débil pero más consistente que en versiones anteriores. La granularidad fina apunta a X[3.70-4.50) como sub-rango positivo (+3.16% in-sample, 5/10) y la Fase 2 OOS lo confirma con +4.95% en 2020-2024 (419 bets, 2/5 temporadas positivas pero con fuerte ganancia agregada). "Solo X[3.50-4.50)" también sale positivo OOS (+2.50%, 4/5). Aun así, el hit rate real del empate (~26%) sigue siendo casi igual a la implícita media: el yield positivo no viene de acertar más, sino de combinar cuotas altas con una selección por EV/ratio que descarta apuestas malas.

| Experimento | Resultado |
|---|---|
| Exp 1: BASE sin X (2015-2024) | -3.71% ROI, 4/10 temporadas, 449 bets |
| Exp 1: BASE + X[3.50-4.50) | -1.18% ROI, 5/10, 886 bets |
| Exp 1: solo X todos los rangos | -2.37% ROI, 5/10, 654 bets |
| Exp 1: sin filtro (solo EV>5%) | -7.62% ROI, 1/10, 3394 bets |
| Exp 2: mejor sub-rango (X[3.70-4.50)) | +3.16% ROI, 5/10, 739 bets |
| Exp 2: Solo X[3.50-4.50) | +1.42% ROI, 5/10, 437 bets, hit 26.3% |
| Exp 3: OOS BASE sin X (Fase 2) | -3.42% ROI, 2/5, 190 bets |
| Exp 3: OOS BASE + X[3.50-4.50) | +0.42% ROI, 2/5, 540 bets |
| Exp 3: OOS Solo X[3.50-4.50) | +2.50% ROI, 4/5, 350 bets |
| Exp 3: OOS BASE + X[3.70-4.50) | +4.95% ROI, 2/5, 419 bets |
| Hit real del empate en X[3.50-4.50) | 26.3%, similar a la implícita ~26% |

### Limitaciones

- FIRST_TEST=2015 (10 temporadas) frente al backtest principal (FIRST_TEST=2012, 13 temporadas). No comparable directamente. Es una inconsistencia heredada que conviene unificar antes de cualquier decisión operativa.
- Las muestras de los sub-rangos son pequeñas (n entre 350 y 950). Cualquier ROI puntual de +3% a +5% entra dentro del intervalo de confianza del 0.
- Que el hit real del empate (~26%) coincida con la implícita (~26%) significa que no hay edge probabilístico explotable. El yield positivo viene de combinar cuotas altas con selección por EV/ratio.

### Conclusión operativa

Los resultados sí respaldan considerar el sub-rango X[3.70-4.50) como variante. Aporta el yield más alto del bloque tanto in-sample (+3.16%) como OOS (+4.95%), aunque con muestra reducida. "Solo X[3.50-4.50)" también es positiva en OOS (+2.50%, 4/5). El filtro principal del NB03 sigue sin empate por dos razones: tiene más muestra y no depende de un sub-rango estrecho. Pero el grid search del notebook 09 retoma esta línea y encuentra que `solo_X[3.50-4.50)` es exactamente lo que aparece en las combinaciones robustas.

Antes de adoptar el empate operativamente conviene:

1. Re-evaluar X[3.70-4.50) en 2012-2024 (no solo 2015-2024) para descartar artefacto temporal.
2. Bootstrap del ROI por temporada con la inclusión de X.
3. Validar con seed alternativo y hold-out 2025+ limpio.

Referencia: el backtest principal (notebook 03, 2012-2024, sliding 5, kill switch) deja Flat ROI +2.59% (344 apuestas, 5/13 temporadas positivas) con calibración isotónica vía `TimeSeriesSplit`.